# Coleta de Dados — Pipeline completo da Câmara dos Deputados

Este notebook **reúne, em um só lugar e na ordem de execução**, toda a coleta de dados do
projeto, feita ao vivo da **API pública de Dados Abertos da Câmara**
(`https://dadosabertos.camara.leg.br`, sem chave e sem login). **Não usamos nenhuma base pronta.**

Está organizado em quatro seções, que devem ser rodadas **de cima para baixo, nesta ordem**
(cada seção depende da anterior):

| Seção | O que coleta | Arquivo gerado |
|-------|--------------|----------------|
| 1 | Proposições de Lei (PLs) + seus **temas** (os rótulos) | `proposicoes_temas.csv` (+ `coleta_manifesto.txt`) |
| 2 | Os **513 parlamentares** atuais | `parlamentares.csv` |
| 3 | Os **autores** de cada proposição (vínculo PL → deputado) | `proposicoes_parlamentares.csv` |
| 4 | Os **discursos** de cada parlamentar | `discursos/deputado_*.csv` + `discursos_todos.csv` |

> A ordem importa: a Seção 3 usa o `parlamentares.csv` (Seção 2) e o `proposicoes_temas.csv`
> (Seção 1); a Seção 4 usa o `parlamentares.csv` (Seção 2).


## Como rodar

1. Instale as bibliotecas (uma vez só):
   ```
   pip install requests pandas
   ```
2. Rode as células **de cima para baixo** (Shift+Enter em cada uma), seção por seção.
3. Tempo total aproximado: Seção 1 ~5 min · Seção 2 segundos · Seção 3 ~10 min ·
   Seção 4 ~20–40 min. As Seções 1 e 4 geram os arquivos mais pesados.


## Preparação — carregar as ferramentas

Importamos as bibliotecas e criamos uma função simples, a `chamar_api`, que conversa com a
API e devolve a resposta. Se a internet falhar, ela tenta de novo antes de desistir.

In [ ]:
import os
os.makedirs("dados", exist_ok=True)
import requests   # para acessar a API pela internet
import time       # para dar pequenas pausas entre as chamadas
import pandas as pd  # para organizar os dados em tabela

BASE = "https://dadosabertos.camara.leg.br/api/v2"

def chamar_api(caminho, parametros=None):
    """Faz uma chamada na API e devolve o resultado (em formato JSON).
    Tenta ate 3 vezes caso o servidor falhe."""
    for tentativa in range(3):
        try:
            resposta = requests.get(
                BASE + caminho,
                params=parametros,
                headers={"Accept": "application/json"},
                timeout=30,
            )
            if resposta.status_code == 200:
                return resposta.json()
        except requests.RequestException:
            pass
        time.sleep(1)   # espera 1 segundo e tenta de novo
    return None

print("Pronto! A funcao chamar_api esta carregada.")

# Seção 1 — Proposições de Lei (PLs) e seus temas

Este notebook **baixa proposições de lei (PLs)** da API pública da Câmara dos Deputados,
já com os **temas** que cada proposição recebeu (que serão os rótulos do classificador),
e salva tudo em um arquivo `proposicoes_temas.csv`.

- **Fonte:** API de Dados Abertos da Câmara — `https://dadosabertos.camara.leg.br` (pública, sem login).
- **Período:** 2023 a 2026.
- **Importante:** os dados são coletados aqui, ao vivo, da fonte oficial. **Não usamos nenhuma base pronta.**

> Uma proposição pode ter **mais de um tema** ao mesmo tempo (ex.: "Saúde" e "Direitos Humanos").
> Por isso guardamos **todos** os temas de cada proposição (classificação *multirrótulo*).

## Passo 1 — Baixar a lista de temas

Primeiro pegamos a lista oficial de **temas** que a Câmara usa. Cada tema é um possível
rótulo de uma proposição. A tabela abaixo mostra o código (`cod`) e o nome de cada um.

In [4]:
resposta = chamar_api("/referencias/proposicoes/codTema")
temas = resposta["dados"]

print(f"A API tem {len(temas)} temas. Cada um e um rotulo possivel.\n")

# salva a lista de temas (os rotulos) em um arquivo CSV
tabela_temas = pd.DataFrame(temas)[["cod", "nome"]]
tabela_temas.to_csv("dados/temas.csv", index=False, encoding="utf-8")
print("Arquivo salvo: dados/temas.csv")

tabela_temas

A API tem 32 temas. Cada um e um rotulo possivel.

Arquivo salvo: temas.csv


,cod,nome
0,34,Administração Pública
1,35,"Arte, Cultura e Religião"
2,37,Comunicações
3,39,Esporte e Lazer
4,40,Economia
5,41,Cidades e Desenvolvimento Urbano
6,42,Direito Civil e Processual Civil
7,43,Direito Penal e Processual Penal
8,44,Direitos Humanos e Minorias
9,46,Educação


## Passo 2 — Baixar as proposições de cada tema

Agora a parte principal. Para **cada tema**, pedimos à API as proposições daquele tema,
**ano a ano**. A API entrega os resultados em "páginas" de 100; então pedimos página 1,
2, 3... **até não vir mais nada**.

Cada proposição já vem com a sua **ementa** (um resumo curto do que ela trata) — esse será
o texto que o modelo vai ler. Guardamos uma linha para cada par *proposição + tema*.

> Vai aparecer o nome de cada tema e quantas proposições ele trouxe. **Leva ~5 minutos.**

In [ ]:
ANOS = [2023, 2024, 2025, 2026]   # periodo que vamos coletar

linhas = []   # cada item da lista sera uma proposicao com UM dos seus temas
for tema in temas:
    quantidade = 0
    for ano in ANOS:
        pagina = 1
        while True:
            resposta = chamar_api("/proposicoes", {
                "siglaTipo": "PL",        # somente Projetos de Lei
                "ano": ano,
                "codTema": tema["cod"],   # filtra por este tema
                "itens": 100,             # 100 proposicoes por pagina
                "pagina": pagina,
                "ordem": "ASC",
                "ordenarPor": "id",
            })
            # se nao veio resposta ou a pagina veio vazia, acabou este tema/ano
            if not resposta or not resposta["dados"]:
                break
            for p in resposta["dados"]:
                linhas.append({
                    "id": p["id"],
                    "ementa": (p.get("ementa") or "").strip(),
                    "tema": tema["nome"],
                    "ano": ano,
                    "tipo": "PL",
                })
                quantidade += 1
            pagina += 1
            time.sleep(0.3)   # pequena pausa para nao sobrecarregar o servidor
    print(f"{tema['nome']}: {quantidade}")

print(f"\nTotal de pares proposicao-tema coletados: {len(linhas)}")

## Passo 3 — Juntar os temas de cada proposição

No passo anterior, uma proposição com 3 temas virou 3 linhas separadas. Aqui nós
**juntamos** tudo: cada proposição vira **uma única linha**, com todos os seus temas
escritos juntos, separados por `|` (ex.: `"Saúde|Direitos Humanos e Minorias"`).

Também removemos proposições que vieram **sem texto** de ementa (não servem para o modelo).

In [ ]:
dados = pd.DataFrame(linhas)

# remove proposicoes sem texto de ementa
dados = dados[dados["ementa"].str.len() > 0]

# funcao que junta os temas de uma proposicao em um texto so
def juntar_temas(serie):
    return "|".join(sorted(set(serie)))

# agrupa por 'id': uma linha por proposicao
proposicoes = dados.groupby("id").agg(
    ementa=("ementa", "first"),
    ano=("ano", "min"),
    tipo=("tipo", "first"),
    temas=("tema", juntar_temas),
).reset_index()

media = proposicoes["temas"].str.split("|").map(len).mean()
print(f"Proposicoes unicas: {len(proposicoes)}")
print(f"Em media, cada proposicao tem {media:.2f} temas.\n")
proposicoes.head()

## Passo 4 — Salvar o arquivo de dados

Salvamos o resultado em `proposicoes_temas.csv`. Esse é o **dataset** que vamos usar para
treinar o classificador depois. No fim, mostramos quantas proposições têm cada tema —
útil para entender o tamanho de cada rótulo.

In [ ]:
from collections import Counter

# salva o arquivo na mesma pasta do notebook
proposicoes.to_csv("dados/proposicoes_temas.csv", index=False, encoding="utf-8")
print("Arquivo salvo: dados/proposicoes_temas.csv\n")

# conta quantas proposicoes tem cada tema
contagem = Counter()
for t in proposicoes["temas"]:
    for tema in t.split("|"):
        contagem[tema] += 1

print("Quantas proposicoes tem cada tema:")
for tema, n in contagem.most_common():
    print(f"  {tema}: {n}")

## Passo 5 — Registrar a coleta (manifesto)

Por fim, escrevemos um pequeno arquivo `coleta_manifesto.txt` que registra **quando** e
**como** a coleta foi feita: a data/hora, a fonte, os parâmetros e os totais. Ele também
guarda uma "impressão digital" (SHA-256) do CSV — um código que muda se o arquivo mudar.

Esse manifesto documenta a sua coleta de forma reprodutível: ao rodar este notebook e
versionar os arquivos no seu GitHub, fica registrado que **foi você quem coletou os dados**.

In [ ]:
import hashlib
import datetime

# calcula a "impressao digital" (SHA-256) do arquivo CSV
with open("dados/proposicoes_temas.csv", "rb") as f:
    impressao_digital = hashlib.sha256(f.read()).hexdigest()

texto = f"""MANIFESTO DA COLETA
Gerado em: {datetime.datetime.now():%Y-%m-%d %H:%M:%S}
Fonte: API de Dados Abertos da Camara dos Deputados ({BASE})
Tipo de proposicao: PL (Projeto de Lei)
Anos coletados: {ANOS}
Temas (rotulos) disponiveis: {len(temas)}
Pares proposicao-tema coletados: {len(linhas)}
Proposicoes unicas no CSV: {len(proposicoes)}
Media de temas por proposicao: {media:.2f}
Arquivo gerado: dados/proposicoes_temas.csv
SHA-256 do arquivo: {impressao_digital}
"""

with open("dados/coleta_manifesto.txt", "w", encoding="utf-8") as f:
    f.write(texto)

print(texto)

# Seção 2 — Parlamentares (os 513 deputados atuais)

Este notebook **baixa os parlamentares** da API pública da Câmara dos Deputados, e salva tudo em um arquivo `parlamentares.csv`.

- **Fonte:** API de Dados Abertos da Câmara — `https://dadosabertos.camara.leg.br` (pública, sem login).
- **Período:** 2023 a 2026.
- **Importante:** os dados são coletados aqui, ao vivo, da fonte oficial. **Não usamos nenhuma base pronta.**


In [ ]:
resposta = chamar_api("/deputados")
deputados = resposta["dados"]

print(f"A API tem {len(deputados)} deputados.\n")
parlamentares = pd.DataFrame(deputados)[["id", "nome", "siglaPartido", "siglaUf", "urlFoto"]]

parlamentares.to_csv("dados/parlamentares.csv", index=False, sep=";", encoding="utf-8")


# Seção 3 — Autores das proposições (vínculo PL → deputado)

Este notebook liga cada **proposição (PL)** ao(s) **deputado(s)** que a assinaram, e salva em
`proposicoes_parlamentares.csv`. Esse vínculo é pré-requisito da Fase 6 (cruzar discursos ×
proposições por deputado).

- **Fonte:** API de Dados Abertos da Câmara — `https://dadosabertos.camara.leg.br` (pública, sem login).
- **Como funciona:** em vez de perguntar o autor de cada uma das ~19 mil proposições (lento),
  invertemos a busca — para cada **deputado atual** pedimos as PLs que ele assinou, via o
  parâmetro `idDeputadoAutor`. Mais rápido (~10 min) e sem depender de casar nomes.
- **Decisão:** só consultamos os **513 deputados atuais**; autores de mandato encerrado e
  comissões/órgãos não entram (não teremos discursos deles). A célula final reporta a
  **cobertura** (% das PLs assinadas por deputado atual).
- **Importante:** os dados são coletados ao vivo da fonte oficial. **Não usamos base pronta.**

> Pré-requisitos na pasta: `parlamentares.csv` (gerado por `02_coleta_parlamentares.ipynb`) e
> `proposicoes_temas.csv` (gerado por `01_coleta_dados.ipynb`).

In [ ]:
# Em vez de perguntar "quem e o autor?" para cada uma das ~19 mil proposicoes
# (lento: ~2,7h), invertemos a busca: para cada deputado ATUAL, pedimos as PLs
# que ele assinou, usando o parametro idDeputadoAutor. Sao ~2 mil chamadas (~10 min).
#
# Consequencia (decisao do projeto): so consultamos os 513 deputados atuais, entao
# autores de mandato encerrado e comissoes/orgaos nao entram -- e o que queremos,
# pois nao teremos discursos deles para o cruzamento.

deps = pd.read_csv("dados/parlamentares.csv", sep=";", dtype=str)
deps["nome"] = deps["nome"].str.strip()

# so guardamos pares cuja proposicao esta na nossa base com tema (alinhamento garantido)
props_validas = set(pd.read_csv("dados/proposicoes_temas.csv", dtype=str)["id"])

ANOS = [2023, 2024, 2025, 2026]

linhas = []   # cada item: uma PL assinada por um deputado atual
for n, (_, dep) in enumerate(deps.iterrows(), start=1):
    for ano in ANOS:
        pagina = 1
        while True:
            r = chamar_api("/proposicoes", {
                "idDeputadoAutor": dep["id"],
                "siglaTipo": "PL",
                "ano": ano,
                "itens": 100,
                "pagina": pagina,
                "ordem": "ASC",
                "ordenarPor": "id",
            })
            # sem resposta ou pagina vazia => acabou este deputado/ano
            if not r or not r["dados"]:
                break
            for p in r["dados"]:
                pid = str(p["id"])
                if pid in props_validas:
                    linhas.append({
                        "proposicao_id": pid,
                        "id_autor": dep["id"],
                        "nome_autor": dep["nome"],
                    })
            pagina += 1
            time.sleep(0.2)   # pausa para nao sobrecarregar o servidor
    if n % 50 == 0:
        print(f"  ... {n}/{len(deps)} deputados processados ({len(linhas)} pares ate agora)")

df_autores_proposicoes = pd.DataFrame(linhas).drop_duplicates()
df_autores_proposicoes.to_csv("dados/proposicoes_parlamentares.csv",
                              index=False, sep=";", encoding="utf-8")
print(f"\nTotal de pares (proposicao, deputado atual): {len(df_autores_proposicoes)}")
print("Arquivo salvo: dados/proposicoes_parlamentares.csv")

In [ ]:
# Cobertura e checagem de plausibilidade
total_props = len(props_validas)
props_cobertas = df_autores_proposicoes["proposicao_id"].nunique()
print(f"Proposicoes com tema na base....: {total_props}")
print(f"Proposicoes com >=1 autor atual.: {props_cobertas} "
      f"({100*props_cobertas/total_props:.1f}% de cobertura)")
print(f"Deputados atuais com >=1 PL.....: {df_autores_proposicoes['id_autor'].nunique()} de 513")

print("\nTop 10 deputados por nº de PLs assinadas (checagem de plausibilidade):")
top = (df_autores_proposicoes.groupby("nome_autor")["proposicao_id"]
       .nunique().sort_values(ascending=False).head(10))
print(top.to_string())

# Seção 4 — Discursos dos parlamentares

Este notebook **baixa os discursos de cada deputado** da API pública da Câmara e salva
**um arquivo por deputado** em `discursos/deputado_{id}.csv`. No fim, junta tudo em um único
`discursos_todos.csv`. Esses textos são a entrada da Fase 6 (classificar os discursos por
tema com o BERTimbau e cruzar com as proposições).

- **Fonte:** API de Dados Abertos da Câmara — `https://dadosabertos.camara.leg.br` (pública, sem login).
- **Período:** 2023 a 2026.
- **Um arquivo por deputado** deixa a coleta **retomável**: se parar no meio, ao rodar de
  novo ele **pula** quem já foi baixado.
- Guardamos só os discursos **com texto** (`transcricao` não-vazia) — é o que vamos classificar.
- **Importante:** os dados são coletados ao vivo da fonte oficial. **Não usamos base pronta.**

> Pré-requisito na pasta: `parlamentares.csv` (gerado por `02_coleta_parlamentares.ipynb`).
> Estimativa: ~20–40 min e dezenas de milhares de discursos.

In [ ]:
from datetime import date
import os

# garante a pasta de saida (nao quebra se ja existir)
os.makedirs("dados/discursos", exist_ok=True)

# colunas fixas: garante que ate deputado SEM discurso gere um arquivo com cabecalho
# (evita o EmptyDataError na hora de juntar)
COLS = ["id_deputado", "nome_deputado", "data", "tipo", "sumario", "transcricao"]

deputados = pd.read_csv("dados/parlamentares.csv", sep=";", encoding="utf-8", dtype=str)
deputados["id"] = deputados["id"].str.replace(r"\.0$", "", regex=True)

ANOS = [2023, 2024, 2025, 2026]

for i, dep in deputados.iterrows():
    caminho = f"dados/discursos/deputado_{dep['id']}.csv"
    if os.path.exists(caminho):          # RETOMADA: pula quem ja foi baixado
        continue

    linhas = []
    for ano in ANOS:
        # 2026 ainda esta em curso: vai ate hoje
        data_fim = date.today().isoformat() if ano == 2026 else f"{ano}-12-31"
        pagina = 1
        while True:
            resposta = chamar_api(f"/deputados/{dep['id']}/discursos", {
                "dataInicio": f"{ano}-01-01",
                "dataFim": data_fim,
                "itens": 100,
                "pagina": pagina,
                "ordem": "ASC",
                "ordenarPor": "dataHoraInicio",
            })
            # sem resposta ou pagina vazia => acabou este deputado/ano
            if not resposta or not resposta["dados"]:
                break
            for d in resposta["dados"]:
                texto = (d.get("transcricao") or "").strip()
                if texto:                  # so guarda discurso COM texto (o que vamos classificar)
                    linhas.append({
                        "id_deputado": dep["id"],
                        "nome_deputado": dep["nome"],
                        "data": d.get("dataHoraInicio"),
                        "tipo": d.get("tipoDiscurso"),
                        "sumario": (d.get("sumario") or "").strip(),
                        "transcricao": texto,
                    })
            pagina += 1
            time.sleep(0.3)   # pausa para nao sobrecarregar o servidor

    # columns=COLS garante cabecalho mesmo quando linhas == [] (deputado sem discurso)
    pd.DataFrame(linhas, columns=COLS).to_csv(caminho, sep=";", index=False, encoding="utf-8")
    print(f"{dep['nome']}: {len(linhas)} discursos com texto")

print("\nColeta concluida. Arquivos em dados/discursos/")

In [ ]:
# Juntar todos os arquivos por deputado em um unico dados/discursos_todos.csv
import glob

partes = [pd.read_csv(f, sep=";", dtype=str) for f in glob.glob("dados/discursos/deputado_*.csv")]
discursos = pd.concat([p for p in partes if len(p)], ignore_index=True)
discursos.to_csv("dados/discursos_todos.csv", sep=";", index=False, encoding="utf-8")

print(f"Total de discursos com texto: {len(discursos)}")
print(f"Deputados com >=1 discurso..: {discursos['id_deputado'].nunique()} de 513")
print("\nTop 10 deputados por nº de discursos (checagem de plausibilidade):")
print(discursos.groupby("nome_deputado").size().sort_values(ascending=False).head(10).to_string())